# SiEBERT baseline

English-specialised polarity model `siebert/sentiment-roberta-large-english` (Hartmann et al., 2023), the model used by Charfaoui and Mussard (2024).

**Run in Google Colab with a GPU**: menu *Runtime → Change runtime type → T4 GPU*. Upload `marakech.csv`, then run all cells (about 10–20 minutes). Result: `siebert_predictions.csv`. No API key is needed.

In [ ]:
%pip install -q transformers torch pandas tqdm

In [ ]:
import pandas as pd, torch, transformers, platform, json
from transformers import pipeline
from tqdm.auto import tqdm
MODEL = 'siebert/sentiment-roberta-large-english'
df = pd.read_csv('marakech.csv'); assert len(df) == 14838
full = df['text'].astype(str).str.strip().tolist()
trunc = [t[:500] for t in full]
dev = 0 if torch.cuda.is_available() else -1
print('GPU:', torch.cuda.get_device_name(0) if dev == 0 else 'NO GPU - will be very slow')
clf = pipeline('sentiment-analysis', model=MODEL, device=dev, truncation=True, max_length=512)

In [ ]:
def run(texts, bs=32):
    lab, ppos = [], []
    for i in tqdm(range(0, len(texts), bs)):
        for r in clf(texts[i:i + bs], top_k=None):
            d = {x['label']: x['score'] for x in r}
            ppos.append(d['POSITIVE']); lab.append('Positive' if d['POSITIVE'] >= 0.5 else 'Negative')
    return lab, ppos
lf, pf = run(full); lt, pt = run(trunc)
out = pd.DataFrame({'row_id': range(len(df)), 'siebert_full': lf, 'siebert_full_ppos': pf,
                    'siebert_trunc': lt, 'siebert_trunc_ppos': pt})
out.to_csv('siebert_predictions.csv', index=False)
json.dump({'model': MODEL, 'transformers': transformers.__version__, 'torch': torch.__version__,
           'python': platform.python_version(), 'max_length': 512, 'trunc_chars': 500}, open('siebert_run_info.json', 'w'), indent=2)
print(out.siebert_full.value_counts()); print('DONE -> download siebert_predictions.csv and siebert_run_info.json')